# Fused GPU K-Means: dimension sweep

We use `||x-c||² = ||x||² - 2 x·c + ||c||²` and fuse dot scores with argmin, avoiding an N×K distance matrix in the iterative assignment step. This custom CUDA kernel is separate from the standalone cuBLAS comparison.

Parts A-D demonstrate one N/K/D case. Part E runs `FEATURE_DIMS` across the batch. See [readme.md](readme.md) for the upload workflow and [design notes](docs/IMPLEMENTATION_DESIGN.md) for implementation details.


In [ ]:
!nvidia-smi


In [ ]:
import os
import gc
import math
import time
import shutil
import numpy as np
import pandas as pd
import cupy as cp
from numba import cuda, float64, int32
import matplotlib.pyplot as plt


## A. Data and configuration

We load one case for the demonstrations; batch settings remain independent.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_FOLDER = "/content/drive/MyDrive/kmeans_data"
os.chdir(DATA_FOLDER)

print("Current folder :", os.getcwd())
print("Folders present:", sorted(f for f in os.listdir() if f.startswith("rows")))


In [ ]:
ROWS = 10_000
K = 3

THREADS_PER_BLOCK = 256
BLOCK_SIZE = (16, 16)
TILE_SIZE = 16
MAX_K = 64
MAX_D = 32

INIT_DIR = "results_gpu_atomic_v3"
REQUIRE_CANONICAL_INIT = True

REPEATS = 10

SAMPLE = 100000

file_path = f"rows{ROWS}/synthetic_data_{ROWS}_rows_10_cols_{K}_clusters.csv"
print("File  :", file_path)
print("Exists:", os.path.exists(file_path))
from pathlib import Path

# Select the first D named features; each dataset keeps all its rows.
FEATURE_DIMS = [2, 4, 9]

SWEEP_CASES = None

CANONICAL_INIT_EXTRA_FOLDERS = ["results_gpu_atomic_v3 2"]

def resolve_canonical_initial_path(primary_directory, base_tag):
    primary = Path(primary_directory)
    folders = [primary] + [primary.parent / name for name in CANONICAL_INIT_EXTRA_FOLDERS]
    filename = f"initial_centroids_{base_tag}.csv"
    candidates = [folder / filename for folder in folders]
    for path in candidates:
        if path.exists():
            return path
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Full-9D canonical initialization missing; searched: {searched}")

def select_feature_columns(available, dimension):
    if isinstance(dimension, (bool, np.bool_)) or not isinstance(dimension, (int, np.integer)):
        raise ValueError("Feature dimensions must be integers")
    if not 1 <= dimension <= 9:
        raise ValueError(f"D={dimension}: existing data supports dimensions 1 through 9")
    selected = [f"col_{i}" for i in range(1, int(dimension) + 1)]
    missing = [c for c in selected if c not in available]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    return selected

def dimension_tag(base_tag, dimension):
    return f"{base_tag}_D{int(dimension)}"

if not FEATURE_DIMS or len(set(FEATURE_DIMS)) != len(FEATURE_DIMS):
    raise ValueError("FEATURE_DIMS must be nonempty and contain no duplicates")
for _dimension in FEATURE_DIMS:
    select_feature_columns([f"col_{i}" for i in range(1, 10)], _dimension)
print("Dimension sweep:", FEATURE_DIMS, "| (N, K) filter:", SWEEP_CASES)

FEATURE_DIM = 9
select_feature_columns([f"col_{i}" for i in range(1, 10)], FEATURE_DIM)


In [ ]:
def load_data(rows, k, feature_dim=9):
    path = f"rows{rows}/synthetic_data_{rows}_rows_10_cols_{k}_clusters.csv"

    local_path = "/content/" + os.path.basename(path)
    if not os.path.exists(local_path):
        print("Copying file to local disk...")
        shutil.copy(path, local_path)
    columns = pd.read_csv(local_path, nrows=0).columns
    feature_cols = select_feature_columns(columns, feature_dim)
    df = pd.read_csv(local_path, usecols=feature_cols, dtype=np.float64)
    return np.ascontiguousarray(df[feature_cols].to_numpy(dtype=np.float64))

print(f"Loading {ROWS:,} rows, D={FEATURE_DIM} from Drive...")
start = time.time()
data_cpu = load_data(ROWS, K, FEATURE_DIM)
N, D = data_cpu.shape
print(f"Data loaded in {time.time() - start:.2f} s: N={N:,}, D={D}, K={K}")


### Canonical centroids

We project the same full-9D reference rows onto the selected D columns.


In [ ]:
import hashlib

def initial_centroid_sha256(C):

    arr = np.ascontiguousarray(np.asarray(C, dtype=np.float64))
    return hashlib.sha256(arr.tobytes()).hexdigest()

def load_initial_centroids(rows, k, n_features=9):

    tag = f"synthetic_data_{rows}_rows_10_cols_{k}_clusters"
    path = str(resolve_canonical_initial_path(INIT_DIR, tag))

    if not os.path.exists(path):
        raise FileNotFoundError(f"Canonical initial-centroid file not found: {path}; copy the existing full-9D Atomic V3 files first")

    df = pd.read_csv(path)
    if "cluster" in df.columns:
        df = df.drop(columns=["cluster"])
    feature_cols = select_feature_columns(df.columns, n_features)
    centroids = np.ascontiguousarray(df[feature_cols].to_numpy(dtype=np.float64))

    if centroids.shape != (k, n_features):
        raise ValueError(f"{path} has shape {centroids.shape}; expected {(k, n_features)}")
    return centroids, path

centroids_cpu, init_source = load_initial_centroids(ROWS, K, D)
print("Initial centroids from:", init_source)
print("Initial centroid SHA256:", initial_centroid_sha256(centroids_cpu))
print("Centroids shape:", centroids_cpu.shape)


In [ ]:
def compare_results(cpu_result, gpu_result, cpu_time, gpu_time):
    print("\nVerifying the results...")

    if cpu_result.dtype.kind == "f":
        difference = np.abs(cpu_result - gpu_result)
        print("Diagnostic Info:")
        print(f"  - Maximum absolute difference between CPU and GPU results: {np.max(difference):.8f}")
        print(f"  - Average absolute difference between CPU and GPU results: {np.mean(difference):.8f}")
        match = np.allclose(cpu_result, gpu_result)
    else:
        different = np.sum(cpu_result != gpu_result)
        print("Diagnostic Info:")
        print(f"  - Points with a different cluster: {different}")
        match = (different == 0)

    if match:
        print("Results are correct! The GPU and CPU computations match.")
    else:
        print("Error: The results from GPU and CPU do not match!")

    print("\nPerformance Summary:")
    print(f"GPU Time: {gpu_time:.6f} seconds")
    print(f"CPU Time: {cpu_time:.6f} seconds")
    if gpu_time > 0:
        print(f"GPU is approximately {cpu_time / gpu_time:.2f}x faster than the CPU.")


## B. CPU assignment check

We compare direct squared distances with the GEMM identity using fixed centroids.


In [ ]:
print("Executing distance calculation on the CPU (normal way)...")
start_cpu = time.time()

distance_cpu = np.zeros((N, K))
for k in range(K):
    diff = data_cpu - centroids_cpu[k]
    distance_cpu[:, k] = np.sum(diff ** 2, axis=1)

cluster_cpu = np.argmin(distance_cpu, axis=1)

end_cpu = time.time()
cpu_time = end_cpu - start_cpu
print(f"CPU execution complete. Time taken: {cpu_time:.6f} seconds.")
print("First 10 clusters:", cluster_cpu[:10])


In [ ]:
print("Executing distance calculation on the CPU (GEMM way)...")
start_cpu = time.time()

s_cpu = np.dot(data_cpu, centroids_cpu.T)
c_square_cpu = np.sum(centroids_cpu ** 2, axis=1)
score_cpu = c_square_cpu - 2 * s_cpu
cluster_cpu_gemm = np.argmin(score_cpu, axis=1)

end_cpu = time.time()
cpu_gemm_time = end_cpu - start_cpu
print(f"CPU execution complete. Time taken: {cpu_gemm_time:.6f} seconds.")
print("First 10 clusters:", cluster_cpu_gemm[:10])


In [ ]:
compare_results(cluster_cpu, cluster_cpu_gemm, cpu_time, cpu_gemm_time)
print("(Here 'GPU' = CPU GEMM way, 'CPU' = CPU normal way)")

del distance_cpu, s_cpu, score_cpu
_ = gc.collect()


## C. GPU assignment experiments

We compare naive and tiled dot products, custom fused assignment, and a complete CuPy assignment pipeline. Dot-product-only timings exclude argmin.


In [ ]:
print("Transferring data to the GPU...")
start = time.time()

data_gpu = cuda.to_device(data_cpu)
centroids_t_gpu = cuda.to_device(np.ascontiguousarray(centroids_cpu.T))

cuda.synchronize()
print(f"Data transferred to GPU. Time taken: {time.time() - start:.6f} seconds.")

s_cpu_sample = np.dot(data_cpu[:SAMPLE], centroids_cpu.T)


### C1. Naive dot-product matrix


In [ ]:
@cuda.jit
def gemm_naive_kernel(A, B, C):

    row = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    col = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0

        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]

        C[row, col] = total


In [ ]:
s_gpu = cuda.device_array((N, K), dtype=np.float64)

GRID_SIZE = (
    (N + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0],
    (K + BLOCK_SIZE[1] - 1) // BLOCK_SIZE[1]
)

gemm_naive_kernel[GRID_SIZE, BLOCK_SIZE](data_gpu, centroids_t_gpu, s_gpu)
cuda.synchronize()

print("Executing naive GEMM kernel on the GPU...")
times = []
for run in range(REPEATS):
    start_gpu = time.time()
    gemm_naive_kernel[GRID_SIZE, BLOCK_SIZE](data_gpu, centroids_t_gpu, s_gpu)
    cuda.synchronize()
    end_gpu = time.time()
    times.append(end_gpu - start_gpu)

naive_time = min(times)
print(f"GPU execution complete. Time taken: {naive_time:.6f} seconds (fastest of {REPEATS} runs, average {np.mean(times):.6f}).")

s_gpu_result_cpu = s_gpu[:SAMPLE].copy_to_host()
compare_results(s_cpu_sample, s_gpu_result_cpu, cpu_gemm_time, naive_time)


### C2. Shared-memory tiled dot-product matrix


In [ ]:
@cuda.jit
def gemm_tiled_kernel(A, B, C):

    tileA = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float64)
    tileB = cuda.shared.array((TILE_SIZE, TILE_SIZE), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.x * TILE_SIZE + tx
    col = cuda.blockIdx.y * TILE_SIZE + ty

    n_rows = A.shape[0]
    n_inner = A.shape[1]
    n_cols = B.shape[1]

    temp = 0.0

    for m in range((n_inner + TILE_SIZE - 1) // TILE_SIZE):

        if row < n_rows and (m * TILE_SIZE + ty) < n_inner:
            tileA[tx, ty] = A[row, m * TILE_SIZE + ty]
        else:
            tileA[tx, ty] = 0.0

        if col < n_cols and (m * TILE_SIZE + tx) < n_inner:
            tileB[tx, ty] = B[m * TILE_SIZE + tx, col]
        else:
            tileB[tx, ty] = 0.0

        cuda.syncthreads()

        for k in range(TILE_SIZE):
            temp += tileA[tx, k] * tileB[k, ty]

        cuda.syncthreads()

    if row < n_rows and col < n_cols:
        C[row, col] = temp


In [ ]:
GRID_SIZE = (
    (N + TILE_SIZE - 1) // TILE_SIZE,
    (K + TILE_SIZE - 1) // TILE_SIZE
)

gemm_tiled_kernel[GRID_SIZE, (TILE_SIZE, TILE_SIZE)](data_gpu, centroids_t_gpu, s_gpu)
cuda.synchronize()

print("Executing tiled GEMM kernel on the GPU...")
times = []
for run in range(REPEATS):
    start_gpu = time.time()
    gemm_tiled_kernel[GRID_SIZE, (TILE_SIZE, TILE_SIZE)](data_gpu, centroids_t_gpu, s_gpu)
    cuda.synchronize()
    end_gpu = time.time()
    times.append(end_gpu - start_gpu)

tiled_time = min(times)
print(f"GPU execution complete. Time taken: {tiled_time:.6f} seconds (fastest of {REPEATS} runs, average {np.mean(times):.6f}).")

s_gpu_result_cpu = s_gpu[:SAMPLE].copy_to_host()
compare_results(s_cpu_sample, s_gpu_result_cpu, cpu_gemm_time, tiled_time)

del s_gpu, data_gpu


### C3. Fused score and argmin

We use transposed D×N data and shared centroids, retaining only the winning label per point.


In [ ]:
@cuda.jit
def gemm_argmin_fused_kernel(data_t, centroids, c_square, clusters):

    centroids_shared = cuda.shared.array((MAX_K, MAX_D), dtype=float64)
    c_square_shared = cuda.shared.array(MAX_K, dtype=float64)

    tid = cuda.threadIdx.x
    n_clusters = centroids.shape[0]
    n_features = centroids.shape[1]

    if tid < n_clusters:
        for j in range(n_features):
            centroids_shared[tid, j] = centroids[tid, j]
        c_square_shared[tid] = c_square[tid]

    cuda.syncthreads()

    idx = cuda.grid(1)
    if idx < data_t.shape[1]:
        best_cluster = 0
        best_score = math.inf

        for k in range(n_clusters):
            dot = 0.0
            for j in range(n_features):
                dot += data_t[j, idx] * centroids_shared[k, j]

            score = c_square_shared[k] - 2.0 * dot

            if score < best_score:
                best_score = score
                best_cluster = k

        clusters[idx] = best_cluster


In [ ]:
print("Transferring data to the GPU...")
data_t_gpu = cuda.to_device(np.ascontiguousarray(data_cpu.T))
centroids_gpu = cuda.to_device(centroids_cpu)
c_square_gpu = cuda.to_device(np.sum(centroids_cpu ** 2, axis=1))
clusters_gpu = cuda.device_array(N, dtype=np.int32)
print("Data transferred to GPU.")

blocks_per_grid = (N + (THREADS_PER_BLOCK - 1)) // THREADS_PER_BLOCK

gemm_argmin_fused_kernel[blocks_per_grid, THREADS_PER_BLOCK](data_t_gpu, centroids_gpu, c_square_gpu, clusters_gpu)
cuda.synchronize()

print("\nExecuting fused GEMM + argmin kernel on the GPU...")
times = []
for run in range(REPEATS):
    start_gpu = time.time()
    gemm_argmin_fused_kernel[blocks_per_grid, THREADS_PER_BLOCK](data_t_gpu, centroids_gpu, c_square_gpu, clusters_gpu)
    cuda.synchronize()
    end_gpu = time.time()
    times.append(end_gpu - start_gpu)

fused_time = min(times)
print(f"GPU execution complete. Time taken: {fused_time:.6f} seconds (fastest of {REPEATS} runs, average {np.mean(times):.6f}).")

cluster_gpu_result_cpu = clusters_gpu.copy_to_host()
compare_results(cluster_cpu, cluster_gpu_result_cpu, cpu_time, fused_time)


### C4. CuPy / cuBLAS assignment

We materialize dot scores, construct distances and run argmin.


In [ ]:
print("Transferring data to the GPU...")
data_cp = cp.asarray(data_cpu)
centroids_cp = cp.asarray(centroids_cpu)
print("Data transferred to GPU.")

def cupy_assign(data_cp, centroids_cp):
    s_cp = cp.dot(data_cp, centroids_cp.T)
    score_cp = cp.sum(centroids_cp ** 2, axis=1) - 2 * s_cp
    return cp.argmin(score_cp, axis=1)

cupy_assign(data_cp, centroids_cp)
cp.cuda.Stream.null.synchronize()

print("\nExecuting GEMM + argmin on the GPU with CuPy...")
times = []
for run in range(REPEATS):
    start_gpu = time.time()
    cluster_cp = cupy_assign(data_cp, centroids_cp)
    cp.cuda.Stream.null.synchronize()
    end_gpu = time.time()
    times.append(end_gpu - start_gpu)

cupy_time = min(times)
print(f"GPU execution complete. Time taken: {cupy_time:.6f} seconds (fastest of {REPEATS} runs, average {np.mean(times):.6f}).")

cluster_cupy_result_cpu = cp.asnumpy(cluster_cp)
compare_results(cluster_cpu, cluster_cupy_result_cpu, cpu_time, cupy_time)

del data_cp, centroids_cp, cluster_cp
cp.get_default_memory_pool().free_all_blocks()


### C5. Assignment timings

These standalone measurements are separate from the iterative K-Means benchmark.


In [ ]:
flops = N * K * (2 * D + 2)

summary_assign = pd.DataFrame([
    {"Version": "CPU normal",                   "Time (ms)": cpu_time * 1000},
    {"Version": "CPU GEMM (NumPy)",             "Time (ms)": cpu_gemm_time * 1000},
    {"Version": "GPU naive GEMM kernel",        "Time (ms)": naive_time * 1000},
    {"Version": "GPU tiled GEMM kernel",        "Time (ms)": tiled_time * 1000},
    {"Version": "GPU fused GEMM + argmin",      "Time (ms)": fused_time * 1000},
    {"Version": "GPU CuPy GEMM + argmin",       "Time (ms)": cupy_time * 1000},
])
summary_assign["Speed-up vs CPU normal"] = (cpu_time * 1000 / summary_assign["Time (ms)"]).round(1)
summary_assign["GFLOP/s"] = (flops / (summary_assign["Time (ms)"] / 1000) / 1e9).round(1)
summary_assign["Time (ms)"] = summary_assign["Time (ms)"].round(2)

print(f"Dataset: {N:,} rows, K = {K}")
print("Note: naive and tiled = GEMM only; all other rows include the argmin.")
summary_assign
summary_assign["Rows"] = N
summary_assign["Features"] = D
summary_assign["K"] = K
display(summary_assign)


## D. Iterative K-Means

We combine fused assignment with global or shared atomic accumulation. Centroid means and stopping checks remain on the host.


In [ ]:
MAX_ITER = 400
TOL = 1e-4
CHANGE_TOL = 1e-4
STABLE_ITERS = 3

@cuda.jit
def assign_clusters_kernel(data_t, centroids, c_square, clusters, changed):
    centroids_shared = cuda.shared.array((MAX_K, MAX_D), dtype=float64)
    c_square_shared = cuda.shared.array(MAX_K, dtype=float64)
    changed_shared = cuda.shared.array(256, dtype=int32)

    tid = cuda.threadIdx.x
    n_clusters = centroids.shape[0]
    n_features = centroids.shape[1]

    if tid < n_clusters:
        for j in range(n_features):
            centroids_shared[tid, j] = centroids[tid, j]
        c_square_shared[tid] = c_square[tid]

    cuda.syncthreads()

    local_changed = 0
    idx = cuda.grid(1)
    if idx < data_t.shape[1]:
        best_cluster = 0
        best_score = math.inf

        for k in range(n_clusters):
            dot = 0.0
            for j in range(n_features):
                dot += data_t[j, idx] * centroids_shared[k, j]
            score = c_square_shared[k] - 2.0 * dot
            if score < best_score:
                best_score = score
                best_cluster = k

        if clusters[idx] != best_cluster:
            local_changed = 1
        clusters[idx] = best_cluster

    changed_shared[tid] = local_changed
    cuda.syncthreads()

    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            changed_shared[tid] += changed_shared[tid + stride]
        cuda.syncthreads()
        stride //= 2

    if tid == 0 and changed_shared[0] > 0:
        cuda.atomic.add(changed, 0, changed_shared[0])

@cuda.jit
def sum_clusters_kernel(data_t, clusters, sums, counts):
    idx = cuda.grid(1)
    if idx < data_t.shape[1]:
        k = clusters[idx]
        cuda.atomic.add(counts, k, 1)
        for j in range(data_t.shape[0]):
            cuda.atomic.add(sums, (k, j), data_t[j, idx])

@cuda.jit
def sum_clusters_shared_kernel(data_t, clusters, sums, counts):

    sums_shared = cuda.shared.array((MAX_K, MAX_D), dtype=float64)
    counts_shared = cuda.shared.array(MAX_K, dtype=int32)

    tid = cuda.threadIdx.x
    block_size = cuda.blockDim.x
    n_clusters = sums.shape[0]
    n_features = sums.shape[1]

    j = tid
    while j < n_clusters * n_features:
        sums_shared[j // n_features, j % n_features] = 0.0
        j += block_size
    if tid < n_clusters:
        counts_shared[tid] = 0
    cuda.syncthreads()

    idx = cuda.grid(1)
    if idx < data_t.shape[1]:
        k = clusters[idx]
        cuda.atomic.add(counts_shared, k, 1)
        for f in range(n_features):
            cuda.atomic.add(sums_shared, (k, f), data_t[f, idx])
    cuda.syncthreads()

    j = tid
    while j < n_clusters * n_features:
        k = j // n_features
        if counts_shared[k] > 0:
            cuda.atomic.add(sums, (k, j % n_features), sums_shared[k, j % n_features])
        j += block_size
    if tid < n_clusters and counts_shared[tid] > 0:
        cuda.atomic.add(counts, tid, counts_shared[tid])


In [ ]:
def cuda_kmeans(data_t_gpu, centroids_host, sum_method="atomic"):
    n_features, n_points = data_t_gpu.shape
    n_clusters = centroids_host.shape[0]
    if not (0 < n_features <= MAX_D and 0 < n_clusters <= MAX_K):
        raise ValueError(f"Unsupported D={n_features}, K={n_clusters}; limits are {MAX_D}, {MAX_K}")
    if centroids_host.shape[1] != n_features:
        raise ValueError("Data and canonical centroid feature counts differ")
    if sum_method not in {"atomic", "shared"}:
        raise ValueError("sum_method must be 'atomic' or 'shared'")
    centroids = centroids_host.copy()

    threads_per_block = THREADS_PER_BLOCK
    blocks_per_grid = (n_points + (threads_per_block - 1)) // threads_per_block

    clusters_gpu = cuda.to_device(np.full(n_points, -1, dtype=np.int32))
    stable = 0
    history = []
    converged = False
    reason = "max_iterations"

    for iteration in range(1, MAX_ITER + 1):

        centroids_gpu = cuda.to_device(centroids)
        c_square_gpu = cuda.to_device(np.sum(centroids ** 2, axis=1))
        changed_gpu = cuda.to_device(np.zeros(1, dtype=np.int32))

        assign_clusters_kernel[blocks_per_grid, threads_per_block](
            data_t_gpu, centroids_gpu, c_square_gpu, clusters_gpu, changed_gpu)

        sums_gpu = cuda.to_device(np.zeros((n_clusters, n_features)))
        counts_gpu = cuda.to_device(np.zeros(n_clusters, dtype=np.int32))

        if sum_method == "shared":
            sum_clusters_shared_kernel[blocks_per_grid, threads_per_block](
                data_t_gpu, clusters_gpu, sums_gpu, counts_gpu)
        else:
            sum_clusters_kernel[blocks_per_grid, threads_per_block](
                data_t_gpu, clusters_gpu, sums_gpu, counts_gpu)
        cuda.synchronize()

        sums = sums_gpu.copy_to_host()
        counts = counts_gpu.copy_to_host()

        new_centroids = centroids.copy()
        for k in range(n_clusters):
            if counts[k] > 0:
                new_centroids[k] = sums[k] / counts[k]

        shift = float(np.max(np.sqrt(np.sum((new_centroids - centroids) ** 2, axis=1))))
        changed = int(changed_gpu.copy_to_host()[0])
        change_fraction = changed / n_points
        centroids = new_centroids

        if change_fraction <= CHANGE_TOL:
            stable += 1
        else:
            stable = 0

        centroid_met = shift < TOL
        stability_met = stable >= STABLE_ITERS
        history.append({
            "iteration": iteration,
            "assignment_changes": changed,
            "assignment_change_fraction": change_fraction,
            "assignment_change_percent": change_fraction * 100.0,
            "max_centroid_movement": shift,
            "stable_fraction_streak": stable,
            "centroid_tolerance_met": centroid_met,
            "assignment_fraction_tolerance_met": change_fraction <= CHANGE_TOL,
            "assignment_stability_met": stability_met,
        })

        if centroid_met or stability_met:
            converged = True
            if centroid_met and stability_met:
                reason = "both"
            elif stability_met:
                reason = "assignment_fraction_stable"
            else:
                reason = "centroid_tolerance"
            break

    info = {
        "converged": converged,
        "convergence_reason": reason,
        "assignment_changes": changed,
        "assignment_change_fraction": change_fraction,
        "stable_fraction_streak": stable,
        "history": pd.DataFrame(history),
    }

    return centroids, clusters_gpu.copy_to_host(), iteration, shift, info

warmup_start = time.perf_counter()
small_cpu = np.random.default_rng(0).uniform(0, 1, (1000, FEATURE_DIM))
small_gpu = cuda.to_device(np.ascontiguousarray(small_cpu.T))
cuda_kmeans(small_gpu, small_cpu[:3].copy(), sum_method="atomic")
cuda_kmeans(small_gpu, small_cpu[:3].copy(), sum_method="shared")
cuda.synchronize()
WARMUP_TIME = time.perf_counter() - warmup_start
print(f"Kernels compiled. Warm-up: {WARMUP_TIME:.2f} seconds (excluded from timing).")


In [ ]:
print(f"Executing K-Means on the GPU ({N:,} rows, K = {K})...")
start_gpu = time.time()
final_centroids, final_clusters, iterations, shift, info = cuda_kmeans(data_t_gpu, centroids_cpu)
end_gpu = time.time()

kmeans_time = end_gpu - start_gpu
print(f"GPU execution complete. Time taken: {kmeans_time:.6f} seconds.")
print("\nIterations        :", iterations)
print(f"Final shift       : {shift:.6f}")
print("Convergence reason:", info["convergence_reason"])
print(f"Time per iteration: {kmeans_time / iterations * 1000:.2f} ms")
print("Points per cluster:", np.bincount(final_clusters, minlength=K))


### D2. Shared atomic accumulation

We accumulate a K×D table per block before updating global sums; addition order can change floating-point rounding.


In [ ]:
print(f"Executing K-Means on the GPU with the shared-memory sum step ({N:,} rows, K = {K})...")
start_gpu = time.time()
final_centroids_shared, final_clusters_shared, iterations_shared, shift_shared, info_shared = cuda_kmeans(
    data_t_gpu, centroids_cpu, sum_method="shared")
end_gpu = time.time()

kmeans_shared_time = end_gpu - start_gpu
print(f"GPU execution complete. Time taken: {kmeans_shared_time:.6f} seconds.")

print("\nIterations (atomic / shared):", iterations, "/", iterations_shared)
print(f"Time per iteration (atomic): {kmeans_time / iterations * 1000:.2f} ms")
print(f"Time per iteration (shared): {kmeans_shared_time / iterations_shared * 1000:.2f} ms")

compare_results(final_clusters, final_clusters_shared, kmeans_time, kmeans_shared_time)
print("(Here 'CPU' = atomic sum step, 'GPU' = shared-memory sum step)")
print(f"Maximum centroid difference: {np.max(np.abs(final_centroids - final_centroids_shared)):.2e}")


## E. Dimension sweep

We run all configured N/K/D cases with canonical initialization, per-D warm-up outside timing, and dimension-specific resume files. `SUM_METHOD` selects independent atomic or shared-sum output folders.


In [ ]:
for name in ["data_cpu", "data_t_gpu", "centroids_gpu", "c_square_gpu", "clusters_gpu",
             "cluster_cpu", "cluster_cpu_gemm", "cluster_gpu_result_cpu",
             "cluster_cupy_result_cpu", "final_clusters", "s_cpu_sample", "s_gpu_result_cpu"]:
    if name in globals():
        del globals()[name]
_ = gc.collect()

DATASETS = [
    (10000, 3), (10000, 5), (10000, 7),
    (50000, 7), (50000, 9), (50000, 11),
    (100000, 13), (100000, 15), (100000, 17),
    (1000000, 3), (1000000, 5), (1000000, 7),
    (5000000, 7), (5000000, 9), (5000000, 11),
    (10000000, 13), (10000000, 15), (10000000, 17),
]

SUM_METHOD = "atomic"

if SUM_METHOD not in {"atomic", "shared"}:
    raise ValueError("SUM_METHOD must be 'atomic' or 'shared'")
OUT_FOLDER = "results_gpu_gemm_fused_v3_aligned_dimension_sweep"
if SUM_METHOD == "shared":
    OUT_FOLDER += "_shared_sum"
SKIP_COMPLETED = True
HISTORY_FOLDER = f"{OUT_FOLDER}/history"
os.makedirs(HISTORY_FOLDER, exist_ok=True)
SUMMARY_FILE = f"{OUT_FOLDER}/summary.csv"

def calc_wcss(data, centroids, clusters):
    total = 0.0
    for start in range(0, len(data), 1000000):
        diff = data[start:start + 1000000] - centroids[clusters[start:start + 1000000]]
        total += np.sum(diff ** 2)
    return total

import json, platform, numba
gpu_name = cuda.get_current_device().name
ENV = {
    "feature_dimensions": FEATURE_DIMS,
    "sweep_cases": SWEEP_CASES,
    "feature_selection": "first D named col_* features",
    "implementation": "gpu_gemm_fused_v3_aligned",
    "gpu_name": gpu_name.decode() if isinstance(gpu_name, bytes) else gpu_name,
    "compute_capability": ".".join(map(str, cuda.get_current_device().compute_capability)),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "numba": numba.__version__,
    "threads_per_block": THREADS_PER_BLOCK,
    "max_k": MAX_K,
    "max_d": MAX_D,
    "sum_method": SUM_METHOD,
    "centroid_tolerance": TOL,
    "assignment_change_fraction_tolerance": CHANGE_TOL,
    "assignment_stable_iterations": STABLE_ITERS,
    "max_iterations": MAX_ITER,
    "canonical_init_dir": INIT_DIR,
    "jit_warmup_excluded_from_timing": True,
    "data_root": DATA_FOLDER,
}
with open(f"{OUT_FOLDER}/environment.json", "w") as fh:
    json.dump(ENV, fh, indent=2)

results = pd.read_csv(SUMMARY_FILE).to_dict("records") if os.path.exists(SUMMARY_FILE) else []
def persist_fused_row(row):
    global results
    results = [r for r in results if r["Dataset"] != row["Dataset"]] + [row]
    pd.DataFrame(results).sort_values(["Rows", "K", "Features"]).to_csv(SUMMARY_FILE, index=False)

cases = [(rows, k, d) for rows, k in DATASETS for d in FEATURE_DIMS
         if SWEEP_CASES is None or (rows, k) in SWEEP_CASES]
print(f"Run plan: {len(cases)} (CSV, D) cases")

for rows, k, feature_dim in cases:
    source_tag = f"synthetic_data_{rows}_rows_10_cols_{k}_clusters"
    tag = dimension_tag(source_tag, feature_dim)
    final_path = f"{OUT_FOLDER}/final_results_{tag}.csv"
    if SKIP_COMPLETED and os.path.exists(final_path):
        print("SKIP (already completed):", tag)
        persist_fused_row(pd.read_csv(final_path).iloc[0].to_dict())
        continue
    print(f"\n===== {rows:,} rows, K={k}, D={feature_dim} =====")

    data_host = load_data(rows, k, feature_dim)
    n_points = data_host.shape[0]

    centroids_host, init_source = load_initial_centroids(rows, k, feature_dim)
    init_sha = initial_centroid_sha256(centroids_host)

    warmup_start = time.perf_counter()
    warm_host = np.random.default_rng(0).normal(size=(256, feature_dim))
    warm_device = cuda.to_device(np.ascontiguousarray(warm_host.T))
    cuda_kmeans(warm_device, warm_host[:k].copy(), sum_method=SUM_METHOD)
    cuda.synchronize()
    warmup_time = time.perf_counter() - warmup_start
    del warm_host, warm_device

    data_t_device = cuda.to_device(np.ascontiguousarray(data_host.T))
    cuda.synchronize()

    start_gpu = time.perf_counter()
    final_centroids, clusters_host, iterations, shift, info = cuda_kmeans(data_t_device, centroids_host, sum_method=SUM_METHOD)
    gpu_time = time.perf_counter() - start_gpu

    wcss = calc_wcss(data_host, final_centroids, clusters_host)

    row = {
        "Implementation": "gpu_gemm_fused_v3_aligned" if SUM_METHOD == "atomic" else "gpu_gemm_fused_shared_sum_v3_aligned",
        "Initial centroid source": init_source,
        "Initial centroid SHA256": init_sha,
        "Dataset": tag,
        "Source dataset": source_tag,
        "Feature columns": json.dumps(select_feature_columns([f"col_{i}" for i in range(1, 10)], feature_dim)),
        "Rows": n_points,
        "Features": data_host.shape[1],
        "K": k,
        "Execution time": gpu_time,
        "Warmup time": warmup_time,
        "Warmup excluded": True,
        "Iterations": iterations,
        "Per iteration ms": gpu_time / iterations * 1000.0,
        "Converged": info["converged"],
        "Convergence reason": info["convergence_reason"],
        "Final assignment changes": info["assignment_changes"],
        "Final assignment change fraction": info["assignment_change_fraction"],
        "Final assignment change percent": info["assignment_change_fraction"] * 100.0,
        "Stable fraction streak": info["stable_fraction_streak"],
        "Max movement": shift,
        "WCSS": float(wcss),
        "Cluster sizes": json.dumps(np.bincount(clusters_host, minlength=k).tolist()),
        "Centroids": json.dumps(final_centroids.astype(float).tolist()),
    }

    pd.DataFrame([row]).to_csv(final_path, index=False)
    info["history"].to_csv(f"{HISTORY_FOLDER}/history_{tag}.csv", index=False)
    print(f"Iterations: {iterations} ({info['convergence_reason']}), Time taken: {gpu_time:.3f} seconds, "
          f"Per iteration: {row['Per iteration ms']:.2f} ms, SHA256: {init_sha[:12]}...")

    persist_fused_row(row)

    del data_host, data_t_device, clusters_host
    _ = gc.collect()

summary = pd.DataFrame(results)
print("\nSaved:", os.path.abspath(SUMMARY_FILE))
summary


## F. Compare with Atomic

We join on source dataset, N, K and D, validate hashes and feature columns, and report missing counterparts.


In [ ]:
ATOMIC_SUMMARY = "results_gpu_atomic_v3_dimension_sweep/summary.csv"
compare = pd.DataFrame()
if not os.path.exists(ATOMIC_SUMMARY) or not os.path.exists(SUMMARY_FILE):
    print("Run both the Atomic and fused dimension sweeps before comparing.")
else:
    gemm = pd.read_csv(SUMMARY_FILE)
    v3 = pd.read_csv(ATOMIC_SUMMARY)
    keys = ["Source dataset", "Rows", "K", "Features"]
    columns = keys + ["Initial centroid SHA256", "Feature columns", "Iterations", "Execution time", "Per iteration ms", "WCSS"]
    compare = pd.merge(v3[columns], gemm[columns], on=keys, how="outer",
                       suffixes=(" V3", " GEMM"), validate="one_to_one", indicator=True)
    missing = compare[compare["_merge"] != "both"]
    if len(missing):
        print("Unmatched cases (run the missing counterpart):")
        display(missing[keys + ["_merge"]])
    compare = compare[compare["_merge"] == "both"].copy()
    compare = compare.rename(columns={"Execution time V3": "Time V3 (s)",
                                      "Execution time GEMM": "Time GEMM (s)",
                                      "Per iteration ms V3": "ms/iter V3",
                                      "Per iteration ms GEMM": "ms/iter GEMM"})
    if len(compare):
        compare["Same initialization"] = compare["Initial centroid SHA256 V3"] == compare["Initial centroid SHA256 GEMM"]
        compare["Same feature columns"] = compare["Feature columns V3"] == compare["Feature columns GEMM"]
        compare["Comparable"] = compare["Same initialization"] & compare["Same feature columns"]
        compare["Same iterations"] = compare["Iterations V3"] == compare["Iterations GEMM"]
        compare["Same WCSS"] = np.isclose(compare["WCSS V3"], compare["WCSS GEMM"], rtol=1e-9)
        compare["Speed-up"] = np.where(compare["Comparable"], compare["ms/iter V3"] / compare["ms/iter GEMM"], np.nan)
        compare = compare.sort_values(["Rows", "K", "Features"]).reset_index(drop=True)
        print(f"Matched {len(compare)} cases; {compare['Comparable'].sum()} share initialization and features")
        display(compare[keys + ["Comparable", "Same iterations", "Same WCSS", "ms/iter V3", "ms/iter GEMM", "Speed-up"]].round(4))
        compare.to_csv(f"{OUT_FOLDER}/comparison_vs_atomic_dimension_sweep.csv", index=False)
    else:
        print("No matching dimension-sweep cases yet.")


In [ ]:
if len(compare):
    plot_data = compare[compare["Comparable"]].copy()
    if len(plot_data):
        labels = [f"{r:,}\nK={k}, D={d}" for r, k, d in zip(plot_data["Rows"], plot_data["K"], plot_data["Features"])]
        x = np.arange(len(plot_data))
        fig, ax = plt.subplots(figsize=(max(11, len(plot_data) * 0.5), 5))
        ax.bar(x - 0.19, plot_data["ms/iter V3"], 0.38, label="Atomic direct distance")
        ax.bar(x + 0.19, plot_data["ms/iter GEMM"], 0.38, label=f"Fused GEMM ({SUM_METHOD} sums)")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
        ax.set_ylabel("Convergence-run time per iteration (ms)")
        ax.set_title("K-Means dimension sweep: matched initialization and features")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(f"{OUT_FOLDER}/gemm_vs_atomic_v3.png", dpi=150)
        plt.show()


## Interpretation

We compare methods within one N/K/D case. Different feature spaces can change iterations, labels and WCSS. We report timing scopes and avoid claiming one method is always fastest; see [results](docs/EXPERIMENTS_AND_RESULTS.md) and [profiling](docs/PROFILING.md).
